Colonnes nécessaires pour répondre aux deux objectifs

date : pour savoir si la transaction a eu lieu un dimanche ou un autre jour

produit : pour calculer le chiffre d’affaires (CA) par produit

prix_unitaire :pour calculer le montant de chaque ligne

quantite :pour calculer le montant total (prix × quantité)

remise_pct :pour ajuster le montant avec la remise

ville :pour tester la robustesse par ville

moyen_paiement :pour harmoniser les valeurs et corriger les incohérences

Création de l’indicateur is_dimanche et des sous-ensembles

In [56]:
import pandas as pd 
df =pd.read_csv('retail_synthetique_tp3.csv',parse_dates=['date'])
df['is_dimanche']=df['date'].dt.weekday==6
df_dim=df[df['is_dimanche']]
df_autres =df[~df['is_dimanche']]


Mon unité d’analyse est la ligne d’achat (chaque produit acheté dans une transaction).
Ce choix est pertinent car l’objectif est de comparer le chiffre d’affaires par produit entre le dimanche et les autres jours.
Si on prenait le ticket (transaction), on perdrait le détail du produit et on ne pourrait pas identifier les produits dominants le dimanche.

In [41]:
### pretraitement 
## q4
doublons = df. duplicated()
print(doublons )
nb_doublons = df. duplicated().sum()
print("le nombre de les doublons",nb_doublons )

# je supremme les doublons mais reste la ligne qui doublies 
if nb_doublons  > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Doublons supprimés.")
    
nb_doublons = df. duplicated().sum()
print("le nombre de les doublons",nb_doublons )




















0       False
1       False
2       False
3       False
4       False
        ...  
3495    False
3496    False
3497    False
3498    False
3499    False
Length: 3500, dtype: bool
le nombre de les doublons 0
le nombre de les doublons 0


In [44]:
nan =df.isnull().sum()
print(nan)

date                0
ville               0
magasin             0
transaction_id      0
client_id           0
sexe              110
age_client        117
categorie           0
produit             0
prix_unitaire       0
quantite            0
remise_pct        106
moyen_paiement      0
is_dimanche         0
dtype: int64


Les valeurs manquantes dans sexe et age_client n’impactent pas directement l’analyse du chiffre d’affaires ou la comparaison Dimanche/Autres jours.
En revanche, les valeurs manquantes dans remise_pct peuvent fausser légèrement le calcul du montant, mais peuvent être imputées par 0 (absence de remise).
Aucune valeur manquante n’est observée dans les variables critiques (prix_unitaire, quantite, date, produit)

In [46]:
df['sexe']=df['sexe'].fillna('incnnu')
#sexe remplacier par mot inconu 
moy_age =df['age_client'].mean()
df['age_client']=df['age_client'].fillna(moy_age )
# je remplace  les valeures manquantes  par moyen de age 
df['remise_pct']=df['remise_pct'].fillna(0)
print (df )


           date      ville magasin transaction_id client_id sexe  age_client  \
0    2024-06-12  Marrakech       D        T000001     C1177    F        42.0   
1    2024-12-29        Fes       B        T000002     C1067    M        41.0   
2    2024-06-24  Marrakech       A        T000003     C1216    M        27.0   
3    2024-02-26  Marrakech       E        T000004     C1028    F        52.0   
4    2024-04-21  Marrakech       B        T000005     C1654    M        35.0   
...         ...        ...     ...            ...       ...  ...         ...   
3495 2024-04-29      Rabat       A        T003496     C1260    M        31.0   
3496 2024-09-22     Tanger       E        T003497     C1203    M        18.0   
3497 2024-12-09     Tanger       A        T003498     C1004    M        35.0   
3498 2024-01-14      Rabat       B        T003499     C1279    F        26.0   
3499 2024-01-25     Tanger       A        T003500     C1423    M        28.0   

         categorie     produit  prix_un

In [ ]:

paiement_map = {
    'Carte bancaire': 'CB',
    'CB': 'CB',
    'cb': 'CB',
    'Espèces': 'Espèces',
    'Cash': 'Espèces',
    'cash': 'Espèces',
    'Chèque': 'Chèque',
    'cheque': 'Chèque',
    'Virement': 'Virement',
    'virement bancaire': 'Virement'
}

# 2.
df['moyen_paiement'] = df['moyen_paiement'].replace(paiement_map)



           date      ville magasin transaction_id client_id sexe  age_client  \
0    2024-06-12  Marrakech       D        T000001     C1177    F        42.0   
1    2024-12-29        Fes       B        T000002     C1067    M        41.0   
2    2024-06-24  Marrakech       A        T000003     C1216    M        27.0   
3    2024-02-26  Marrakech       E        T000004     C1028    F        52.0   
4    2024-04-21  Marrakech       B        T000005     C1654    M        35.0   
...         ...        ...     ...            ...       ...  ...         ...   
3495 2024-04-29      Rabat       A        T003496     C1260    M        31.0   
3496 2024-09-22     Tanger       E        T003497     C1203    M        18.0   
3497 2024-12-09     Tanger       A        T003498     C1004    M        35.0   
3498 2024-01-14      Rabat       B        T003499     C1279    F        26.0   
3499 2024-01-25     Tanger       A        T003500     C1423    M        28.0   

         categorie     produit  prix_un

In [53]:
# Création de la colonne montant_net
df['montant_net'] = df['prix_unitaire'] * (1 - df['remise_pct'] / 100)

# Arrondir à 2 décimales pour plus de lisibilité
df['montant_net'] = df['montant_net'].round(2)

print("✅ Colonne 'montant_net' créée avec succès !")


✅ Colonne 'montant_net' créée avec succès !


In [1]:
# Fonction pour résumé complet
def resume_stat(df, colonne='montant_net'):
    Q1 = df[colonne].quantile(0.25)
    Q3 = df[colonne].quantile(0.75)
    IQR = Q3 - Q1
    return pd.Series({
        'n': df[colonne].count(),
        'somme': df[colonne].sum(),
        'moyenne': df[colonne].mean(),
        'mediane': df[colonne].median(),
        'Q1': Q1,
        'Q3': Q3,
        'écart-type': df[colonne].std(),
        'IQR': IQR
    })

# Résumé pour Dimanche et Autres jours

resume_autres = resume_stat(df_autres)
resume_dimanche =resume_stat(df_dim)
print("Résumé Dimanche :\n", resume_dimanche)
print("\nRésumé Autres jours :\n", resume_autres)


NameError: name 'df_autres' is not defined